## 35_export_tables  (OLD workspace = Databricks Free Edition) — Tables/Schema Migration step 1 (stream T)

The jobs' tables use 2-part schema.table names on the old workspace (no catalog);
only the model registry and this export volume are catalog-qualified (Free Edition
Unity Catalog has no 2-part form for those).
This exporter snapshots them so the NEW (Azure Databricks) workspace can lift
them with the SCHEMA AND TABLE NAMES UNCHANGED (only the catalog is new;
see runbook Appendix A).
  1. Inventory SHOW TABLES IN <SCHEMA> (the OLD schema name is recorded!)
  2. Snapshot each practice_* table as a Delta folder + row counts — staged
     in an auto-created UC VOLUME (serverless-safe; classic workspaces fall
     back to the driver /tmp path)
  3. Zip inventory + data -> tables_export.zip -> publish for download
PREREQ: run each practice job once in the OLD workspace (Jobs UI -> Run Now)
        so all 5 tables exist: customers, clean, enriched, summary, predictions.


In [ ]:



import os, shutil, json

dbutils.widgets.text("SCHEMA", "practice", "OLD schema name (kept identical after migration)")
dbutils.widgets.text("TABLE_PREFIX", "practice_", "Only export tables starting with this prefix")

SCHEMA = dbutils.widgets.get("SCHEMA").strip()
PREFIX = dbutils.widgets.get("TABLE_PREFIX").strip()


In [ ]:



def table_name(simple_name):
    # 2-part names on the old workspace — no catalog for tables
    return f"{SCHEMA}.{simple_name}"


In [ ]:



# Step 1 — inventory: record the OLD schema name + every practice_* table.
rows = spark.sql(f"SHOW TABLES IN {SCHEMA}").collect()
practice_tables = sorted(
    str(r["tableName"]) for r in rows
    if str(r["tableName"]).startswith(PREFIX) and not bool(r["isTemporary"])
)
assert practice_tables, (
    f"No {PREFIX}* tables found in schema '{SCHEMA}'. Run 00_setup_sample_data first, "
    f"and run each practice job once (runbook Phase 1.3) so the derived tables exist."
)
print(f"OLD schema     : {SCHEMA}")
print(f"Tables to lift : {', '.join(practice_tables)}")


In [ ]:



# Step 2 — snapshot each table as a Delta folder + record row counts.
# Free Edition (serverless) cannot rely on driver-local Spark writes, so on UC
# workspaces the snapshots are staged in an auto-created VOLUME; classic
# workspaces (CE / DBR 12.2) keep the driver /tmp path.
import re

assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA), f"Invalid schema identifier: {SCHEMA!r}"

def _export_root():
    # data_catalog() is the lib's ONLY catalog lookup — volumes are the one
    # platform-forced exception (UC has no 2-part volume path).
    catalog = data_catalog()
    if catalog is None:
        return "/tmp/tables_export"                      # classic fallback
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{SCHEMA}.tables_export_stage")
    print(f"Snapshot volume: {catalog}.{SCHEMA}.tables_export_stage")
    return f"/Volumes/{catalog}/{SCHEMA}/tables_export_stage"

EXPORT_DIR = _export_root()
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

inventory = {"old_schema": SCHEMA, "tables": []}
for t in practice_tables:
    src = table_name(t)
    dst = os.path.join(EXPORT_DIR, "data", t)
    df = spark.read.table(src)
    n = df.count()
    df.write.format("delta").mode("overwrite").save(dst)
    inventory["tables"].append({"table": t, "row_count": n})
    print(f"SNAPSHOT  {src}  ({n} rows)  ->  data/{t}")

with open(os.path.join(EXPORT_DIR, "tables_list.json"), "w") as f:
    json.dump(inventory, f, indent=2)


In [ ]:



# Step 3 — zip and publish for download (same pattern as 30/40).
ZIP_LOCAL = shutil.make_archive("/tmp/tables_export", "zip", EXPORT_DIR)
print(f"Zip ready: {ZIP_LOCAL}  ({os.path.getsize(ZIP_LOCAL)/1024:.1f} KB)")

user = dbutils.notebook.getContext().userName().get()
fname = os.path.basename(ZIP_LOCAL)
try:
    ws_dir = f"/Workspace/Users/{user}/migration_export"
    os.makedirs(ws_dir, exist_ok=True)
    shutil.copy(ZIP_LOCAL, ws_dir)
    print(f"\nDOWNLOAD from the workspace file browser:\n  {ws_dir}/{fname}\n(right-click -> Download)")
except Exception as e:
    print(f"Workspace copy failed ({e}); falling back to DBFS FileStore...")
    dbutils.fs.cp(f"file:{ZIP_LOCAL}", f"dbfs:/FileStore/migration_export/{fname}")
    print(f"Download from: https://<your-old-workspace-url>/files/migration_export/{fname}")
